# 3 - Feature Engineering
Este notebook prepara `./data/titanic_procesado.csv` con exactamente las columnas requeridas.

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, QuantileTransformer, MinMaxScaler
import numpy as np

In [2]:
# 1) Cargar datos
df = pd.read_csv('./data/titanic_clean.csv')
print('Loaded rows:', df.shape[0])

Loaded rows: 891


In [3]:
# 2) Label encode Sex y Embarked (in place)
le_sex = LabelEncoder()
le_emb = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'].astype(str))
df['Embarked'] = le_emb.fit_transform(df['Embarked'].astype(str))
print('Unique Sex:', sorted(df['Sex'].unique()))
print('Unique Embarked:', sorted(df['Embarked'].unique()))

Unique Sex: [np.int64(0), np.int64(1)]
Unique Embarked: [np.int64(0), np.int64(1), np.int64(2)]


In [4]:
# 3) QuantileTransformer on Age and Fare (preserve NaNs by transforming non-null rows)
qt = QuantileTransformer(output_distribution='normal', n_quantiles=500, random_state=42)
for col in ['Age','Fare']:
    mask = df[col].notna()
    if mask.sum()>0:
        vals = df.loc[mask, [col]].astype(float)
        df.loc[mask, col] = qt.fit_transform(vals)[:,0]
print('QuantileTransformer applied to Age and Fare')

QuantileTransformer applied to Age and Fare


In [5]:
# 5) MinMaxScaler on the requested columns (in place)
mms = MinMaxScaler()
cols = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']
# Ensure numeric types
df[cols] = df[cols].astype(float)
df[cols] = mms.fit_transform(df[cols])
print('MinMax scaling applied to:', cols)

MinMax scaling applied to: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [6]:
# 6) Keep Survived unchanged and order columns exactly as requested
expected = ['Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']
if 'Survived' not in df.columns:
    raise ValueError('Survived column not found in source file')
df = df[expected].copy()
print('Final columns:', df.columns.tolist())
print('Shape:', df.shape)

Final columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Shape: (891, 8)


In [7]:
# 8) Save overwriting the processed CSV
df.to_csv('./data/titanic_procesado.csv', index=False)
print('Saved ./data/titanic_procesado.csv with shape', df.shape)

Saved ./data/titanic_procesado.csv with shape (891, 8)
